# Vanilla RAG Lab: Mars Landing Report Q&A

Build a simple RAG system that answers questions about a fictional report of humanity's first Mars landing.

**What you'll learn:**
- Chunk documents with multiple strategies (fixed, recursive, semantic)
- Embed and index chunks into ChromaDB
- Build a RAG agent with LangChain + LangGraph
- Evaluate RAG quality with retrieval and answer metrics
- Improve retrieval with hybrid search (BM25 + dense) and cross-encoder reranking
- Compare vanilla vs hybrid RAG with precision@k, recall@k, nDCG, and LLM-as-judge

In [ ]:
%uv add langchain langchain-openai langchain-chroma chromadb langchain-community langchain-experimental langgraph python-dotenv

In [1]:
import os
import logging
from dotenv import load_dotenv

load_dotenv("../.env")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s - %(message)s",
)
logger = logging.getLogger("rag_lab")

logger.info("OpenAI API key configured: %s", "Yes" if os.getenv("OPENAI_API_KEY") else "No")

2026-04-05 15:16:12,534 [INFO] rag_lab - OpenAI API key configured: Yes


## Step 1: Load the Document

Read the Mars landing report that serves as our knowledge base.

In [2]:
with open("first-time-on-mark.txt", "r") as f:
    document_text = f.read()

logger.info("Loaded document: %d characters", len(document_text))
print(document_text[:500] + "\n...")

2026-04-05 15:16:14,823 [INFO] rag_lab - Loaded document: 8526 characters


**Dateline: July 18, 2050 — Jezero Crater, Mars**

For the first time in human history, the red dust of Mars bears the unmistakable imprint of a human footprint.

At 09:42 Coordinated Mars Time, Commander Elena Navarro descended the final rung of the lander *Astraeus* and stepped onto the Martian surface, pausing only briefly before uttering words that will likely be remembered for generations:

> “We are no longer visitors of the sky—we are a species that has arrived.”

Behind her, the horizon 
...


## Step 2: Chunk the Document

Split the document into smaller chunks for embedding. Three strategies are available — change the `CHUNKING_STRATEGY` flag to compare them:

| Strategy | Description |
|---|---|
| `fixed` | Fixed-size character splits with overlap |
| `recursive` | Recursive splitting that respects natural text boundaries (paragraphs, sections) |
| `semantic` | Groups text by embedding similarity — chunks are semantically coherent |

In [4]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

# =============================================================
#  CHANGE THIS FLAG TO SWITCH CHUNKING STRATEGY
CHUNKING_STRATEGY = "recursive"  # "fixed" | "recursive" | "semantic"
# =============================================================

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100


def chunk_document(text: str, strategy: str) -> list[str]:
    """Split text into chunks using the selected strategy."""
    logger.info("Chunking strategy=%s  size=%d  overlap=%d", strategy, CHUNK_SIZE, CHUNK_OVERLAP)

    if strategy == "fixed":
        splitter = CharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separator="\n",
        )
    elif strategy == "recursive":
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separators=["\n---\n", "\n### ", "\n\n", "\n", " "],
        )
    elif strategy == "semantic":
        embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
        splitter = SemanticChunker(embeddings, breakpoint_threshold_type="percentile")
    else:
        raise ValueError(f"Unknown chunking strategy: {strategy}")

    return splitter.split_text(text)


chunks = chunk_document(document_text, CHUNKING_STRATEGY)
logger.info("Produced %d chunks", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\n{'=' * 60}")
    print(f"Chunk {i}  ({len(chunk)} chars)")
    print(f"{'=' * 60}")
    print(chunk[:300] + ("..." if len(chunk) > 300 else ""))

2026-04-05 15:18:05,976 [INFO] rag_lab - Chunking strategy=recursive  size=500  overlap=100
2026-04-05 15:18:05,976 [INFO] rag_lab - Produced 33 chunks



Chunk 0  (474 chars)
**Dateline: July 18, 2050 — Jezero Crater, Mars**

For the first time in human history, the red dust of Mars bears the unmistakable imprint of a human footprint.

At 09:42 Coordinated Mars Time, Commander Elena Navarro descended the final rung of the lander *Astraeus* and stepped onto the Martian su...

Chunk 1  (445 chars)
Behind her, the horizon stretched in muted tones of rust and amber, broken only by distant ridges and the faint silhouette of ancient riverbeds—silent witnesses to a world that once held water, and perhaps, life. Above, the sky glowed a pale, dusty orange, thinner and more fragile than anything seen...

Chunk 2  (3 chars)
---

Chunk 3  (367 chars)
### The Mission: A Journey Decades in the Making

The mission, officially named *Aurora-1*, is the culmination of over thirty years of international collaboration, technological innovation, and relentless ambition. Unlike previous robotic missions, which paved the way with data and imagery, *Aurora-..

## Step 3: Embed and Index into ChromaDB

Convert chunks into vector embeddings using OpenAI's `text-embedding-3-small` model and store them in a local ChromaDB collection with metadata.

In [ ]:
import chromadb
from langchain_chroma import Chroma
from langchain_core.documents import Document

PERSIST_DIR = "./chroma_db"
COLLECTION_NAME = "mars_landing_report"

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# Clear previous collection so re-runs with different strategies start fresh
chroma_client = chromadb.PersistentClient(path=PERSIST_DIR)
existing = [c.name for c in chroma_client.list_collections()]
if COLLECTION_NAME in existing:
    chroma_client.delete_collection(COLLECTION_NAME)
    logger.info("Deleted existing collection '%s'", COLLECTION_NAME)

documents = [
    Document(
        page_content=chunk,
        metadata={
            "source": "first-time-on-mark.txt",
            "chunk_index": i,
            "chunking_strategy": CHUNKING_STRATEGY,
        },
    )
    for i, chunk in enumerate(chunks)
]

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name=COLLECTION_NAME,
    persist_directory=PERSIST_DIR,
)

logger.info("Indexed %d chunks into ChromaDB (collection='%s')", len(documents), COLLECTION_NAME)

# Quick sanity check
results = vectorstore.similarity_search("Who landed on Mars?", k=3)
print("Sanity check — top 3 results for 'Who landed on Mars?':\n")
for i, doc in enumerate(results):
    print(f"  [{i + 1}] chunk_index={doc.metadata['chunk_index']}")
    print(f"      {doc.page_content}\n")

2026-04-05 15:21:25,868 [INFO] rag_lab - Deleted existing collection 'mars_landing_report'
2026-04-05 15:21:28,935 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-05 15:21:28,987 [INFO] rag_lab - Indexed 33 chunks into ChromaDB (collection='mars_landing_report')
2026-04-05 15:21:29,256 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Sanity check — top 3 results for 'Who landed on Mars?':

  [1] chunk_index=5
      The landing itself was a feat of precision engineering. Mars’ thin atmosphere has long posed a challenge: too thick to ignore, too thin to rely on for safe descent. The *Astraeus* lander employed a hybrid system of heat shielding, supersonic parachutes, and retro-thrusters, guided by AI-assisted navigation that adjusted in real time to atmospheric conditions....

  [2] chunk_index=0
      **Dateline: July 18, 2050 — Jezero Crater, Mars**

For the first time in human history, the red dust of Mars bears the unmistakable imprint of a human footprint.

At 09:42 Coordinated Mars Time, Commander Elena Navarro descended the final rung of the lander *Astraeus* and stepped onto the Martian surface, pausing only briefly before uttering words that will likely be remembered for generations:

> “We are no longer visitors of the sky—we are a species that has arrived.”...

  [3] chunk_index=21
      ### Living on Mars:

## Step 4: Build the RAG Agent

Create a LangGraph agent equipped with a retriever tool that searches the ChromaDB vector store to answer questions.

In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.tools.retriever import create_retriever_tool
from langchain.agents import create_agent

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

retriever_tool = create_retriever_tool(
    retriever,
    name="mars_report_search",
    description=(
        "Search the Mars landing report for information about the Aurora-1 mission, "
        "its crew, scientific findings, spacecraft, and life on Mars. "
        "Use this tool to find facts before answering any question."
    ),
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

SYSTEM_PROMPT = (
    "You are a knowledgeable assistant that answers questions about humanity's "
    "first Mars landing. Always use the mars_report_search tool to retrieve "
    "relevant information before answering. Base your answers strictly on the "
    "retrieved context. If the answer is not in the retrieved documents, say so."
)

agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=SYSTEM_PROMPT,
)

logger.info("RAG agent created with retriever tool")

2026-04-05 15:25:25,422 [INFO] rag_lab - RAG agent created with retriever tool


In [9]:
test_questions = [
    "Who was the first person to step on Mars?",
    "What was the name of the mission and when did it launch?",
    "How many crew members were there and what were their nationalities?",
    "What scientific discoveries were made on Mars?",
    "What challenges did the crew face living on Mars?",
]

for question in test_questions:
    print(f"\n{'=' * 70}")
    print(f"Q: {question}")
    print(f"{'=' * 70}")
    response = agent.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content
    print(f"A: {answer}")


Q: Who was the first person to step on Mars?


2026-04-05 15:25:33,455 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-05 15:25:34,094 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-05 15:25:36,018 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: The first person to step on Mars was Commander Elena Navarro. She made her historic descent from the lander *Astraeus* and stepped onto the Martian surface at 09:42 Coordinated Mars Time on July 18, 2050.

Q: What was the name of the mission and when did it launch?


2026-04-05 15:25:37,460 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-05 15:25:37,661 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-05 15:25:39,301 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: The mission was officially named *Aurora-1*, and it launched in late 2049.

Q: How many crew members were there and what were their nationalities?


...

A: The *Aurora-1* mission crew consists of six members, each from different countries:

1. **Commander Elena Navarro** - Spain
2. **Captain Lucas Andrade** - Brazil
3. **Dr. Hannah Weiss** - Germany

Unfortunately, the report does not provide the names or nationalities of the other three crew members.

Q: What scientific discoveries were made on Mars?


2026-04-05 15:25:52,819 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-05 15:25:53,154 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-05 15:25:59,477 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: The scientific discoveries made on Mars during the Aurora-1 mission include:

1. **Geological Insights**: The terrain of Mars is described as a mosaic of fine dust and jagged rocks, shaped by billions of years of geological processes. The crew has begun collecting samples for analysis, which may reveal more about Mars' geological history.

2. **Potential for Past Habitability**: Preliminary observations suggest that Mars was once more dynamic and potentially more hospitable than previously believed. This raises questions about the planet's past environment and its capacity to support life.

3. **Environmental Challenges**: The crew is studying the extreme conditions on Mars, including temperature fluctuations and radiation exposure. Mars lacks a global magnetic field, making its surface vulnerable to cosmic rays, which is a significant concern for long-term human habitation.

4. **Human Health Studies**: The mission includes monitoring the crew's health to understand how reduced gra

2026-04-05 15:26:01,237 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-05 15:26:01,709 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-05 15:26:07,814 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: The crew faced several significant challenges while living on Mars:

1. **Extreme Temperatures**: Mars experiences dramatic temperature fluctuations, with relatively mild conditions during the day and extreme cold at night.

2. **Radiation Exposure**: The lack of a global magnetic field on Mars leaves the surface vulnerable to cosmic rays, posing a serious health risk. Although the habitat is equipped with shielding, long-term exposure to radiation is a major concern.

3. **Health Monitoring**: The crew's health is under constant observation, with daily assessments conducted by Dr. Weiss to study the effects of reduced gravity and increased radiation on the human body.

4. **Dust Storms**: While the crew had not yet encountered dust storms, these remain a persistent threat that could impact their operations and safety.

5. **Movement in Reduced Gravity**: Mars' gravity is about 38% of Earth's, which affects how the crew moves. They must adapt to longer strides and a different gait w

## Step 5: Evaluate RAG Quality

Evaluate both **retrieval** and **generation** quality using a test dataset with ground-truth answers.

**Retrieval metrics:**
- **Hit Rate** — fraction of questions where the correct context appears in top-k results
- **MRR (Mean Reciprocal Rank)** — average of `1 / rank` for the first relevant chunk

**Answer metrics (LLM-as-judge):**
- **Faithfulness** — does the answer stick to the retrieved context (no hallucination)?
- **Correctness** — does the answer match the ground truth?

In [11]:
eval_dataset = [
    {
        "question": "Who was the commander of the Aurora-1 mission?",
        "ground_truth": "Commander Elena Navarro from Spain.",
        "keywords": ["Elena Navarro", "Spain"],
    },
    {
        "question": "When did humans first land on Mars?",
        "ground_truth": "Humans first landed on Mars on July 18, 2050.",
        "keywords": ["July 18, 2050"],
    },
    {
        "question": "What was the name of the lander?",
        "ground_truth": "The lander was named Astraeus.",
        "keywords": ["Astraeus"],
    },
    {
        "question": "What is the gravity on Mars compared to Earth?",
        "ground_truth": "Mars gravity is about 38 percent of Earth's gravity.",
        "keywords": ["38%", "38 percent"],
    },
    {
        "question": "What was Dr. Malik Okoye's role on the mission?",
        "ground_truth": "Dr. Malik Okoye was the astrobiologist and mission science lead from Nigeria.",
        "keywords": ["astrobiologist", "science lead"],
    },
    {
        "question": "What were the early scientific findings on Mars?",
        "ground_truth": "Preliminary scans identified complex carbon compounds in certain sediment layers.",
        "keywords": ["carbon compounds"],
    },
    {
        "question": "Where did the Aurora-1 mission land on Mars?",
        "ground_truth": "The mission landed at Jezero Crater, believed to be an ancient lakebed.",
        "keywords": ["Jezero Crater"],
    },
    {
        "question": "What was the name of the interplanetary vessel?",
        "ground_truth": "The interplanetary vessel was named Odyssey.",
        "keywords": ["Odyssey"],
    },
    {
        "question": "How long was the journey from Earth to Mars?",
        "ground_truth": "The journey lasted just under seven months.",
        "keywords": ["seven months"],
    },
    {
        "question": "What is the planned total duration of the Aurora-1 mission?",
        "ground_truth": "The mission is scheduled to last approximately 18 months including the return journey.",
        "keywords": ["18 months"],
    },
]

logger.info("Evaluation dataset: %d questions", len(eval_dataset))

2026-04-05 15:27:32,452 [INFO] rag_lab - Evaluation dataset: 10 questions


In [12]:
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

JUDGE_TEMPLATE = """Evaluate the following RAG-generated answer on two criteria.
Score each from 1 (worst) to 5 (best).

Question: {question}
Ground Truth: {ground_truth}
Retrieved Context:
{context}
Generated Answer: {answer}

Criteria:
- Faithfulness: Does the answer ONLY use information present in the Retrieved Context (no hallucination)?
- Correctness: Does the answer correctly address the question when compared to the Ground Truth?

Respond in EXACTLY this format (just the two lines, numbers only):
Faithfulness: <score>
Correctness: <score>"""

eval_results = []

for idx, item in enumerate(eval_dataset):
    question = item["question"]
    ground_truth = item["ground_truth"]
    keywords = item["keywords"]

    # --- Retrieval ---
    retrieved_docs = retriever.invoke(question)
    retrieved_texts = [doc.page_content for doc in retrieved_docs]
    combined_context = "\n---\n".join(retrieved_texts)

    hit = any(
        any(kw.lower() in txt.lower() for txt in retrieved_texts)
        for kw in keywords
    )

    reciprocal_rank = 0.0
    for rank, txt in enumerate(retrieved_texts, start=1):
        if any(kw.lower() in txt.lower() for kw in keywords):
            reciprocal_rank = 1.0 / rank
            break

    # --- Generation ---
    response = agent.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    # --- LLM-as-judge ---
    judge_prompt = JUDGE_TEMPLATE.format(
        question=question,
        ground_truth=ground_truth,
        context=combined_context[:3000],
        answer=answer,
    )
    judge_response = judge_llm.invoke(judge_prompt)
    judge_text = judge_response.content

    faithfulness_score = 0
    correctness_score = 0
    for line in judge_text.strip().split("\n"):
        if line.lower().startswith("faithfulness"):
            try:
                faithfulness_score = int(line.split(":")[1].strip())
            except (ValueError, IndexError):
                pass
        elif line.lower().startswith("correctness"):
            try:
                correctness_score = int(line.split(":")[1].strip())
            except (ValueError, IndexError):
                pass

    eval_results.append({
        "question": question,
        "ground_truth": ground_truth,
        "answer": answer,
        "hit": hit,
        "reciprocal_rank": reciprocal_rank,
        "faithfulness": faithfulness_score,
        "correctness": correctness_score,
    })

    logger.info(
        "[%d/%d] %s  hit=%s  rr=%.2f  faith=%d  correct=%d",
        idx + 1, len(eval_dataset), question[:40],
        hit, reciprocal_rank, faithfulness_score, correctness_score,
    )

logger.info("Evaluation complete")

2026-04-05 15:27:34,906 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-05 15:27:36,123 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-05 15:27:36,431 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-05 15:27:37,654 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-05 15:27:38,239 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-05 15:27:38,247 [INFO] rag_lab - [1/10] Who was the commander of the Aurora-1 mi  hit=True  rr=0.50  faith=5  correct=5
2026-04-05 15:27:38,479 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-05 15:27:39,611 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-05 15:27:39,933 [INFO] httpx - HTTP Request: POST

In [13]:
print(f"\n{'=' * 90}")
print(f"{'EVALUATION RESULTS':^90}")
print(f"{'=' * 90}")
print(f"\n{'Question':<50} {'Hit':>4} {'RR':>5} {'Faith':>6} {'Corr':>6}")
print(f"{'-' * 50} {'-' * 4} {'-' * 5} {'-' * 6} {'-' * 6}")

for r in eval_results:
    q_display = (r['question'][:47] + '...') if len(r['question']) > 50 else r['question']
    hit_symbol = 'Y' if r['hit'] else 'N'
    print(
        f"{q_display:<50} {hit_symbol:>4} {r['reciprocal_rank']:>5.2f} "
        f"{r['faithfulness']:>4}/5  {r['correctness']:>4}/5"
    )

hit_rate = sum(r["hit"] for r in eval_results) / len(eval_results)
mrr = sum(r["reciprocal_rank"] for r in eval_results) / len(eval_results)
avg_faith = sum(r["faithfulness"] for r in eval_results) / len(eval_results)
avg_correct = sum(r["correctness"] for r in eval_results) / len(eval_results)

print(f"\n{'=' * 90}")
print(f"  Chunking Strategy  : {CHUNKING_STRATEGY}")
print(f"  Hit Rate           : {hit_rate:.0%}")
print(f"  MRR                : {mrr:.3f}")
print(f"  Avg Faithfulness   : {avg_faith:.2f} / 5")
print(f"  Avg Correctness    : {avg_correct:.2f} / 5")
print(f"{'=' * 90}")
print(f"\nTip: Re-run the notebook with a different CHUNKING_STRATEGY to compare results.")


                                    EVALUATION RESULTS                                    

Question                                            Hit    RR  Faith   Corr
-------------------------------------------------- ---- ----- ------ ------
Who was the commander of the Aurora-1 mission?        Y  0.50    5/5     5/5
When did humans first land on Mars?                   Y  1.00    5/5     5/5
What was the name of the lander?                      Y  1.00    5/5     1/5
What is the gravity on Mars compared to Earth?        Y  1.00    5/5     5/5
What was Dr. Malik Okoye's role on the mission?       Y  1.00    5/5     5/5
What were the early scientific findings on Mars?      Y  0.20    5/5     5/5
Where did the Aurora-1 mission land on Mars?          N  0.00    5/5     5/5
What was the name of the interplanetary vessel?       Y  1.00    5/5     5/5
How long was the journey from Earth to Mars?          N  0.00    5/5     5/5
What is the planned total duration of the Auror...    Y  1.00 

## Step 6: Hybrid RAG with Reranking

The vanilla RAG above uses only **dense vector retrieval**. We can improve it by combining:

1. **Sparse retrieval (BM25)** — great for exact keyword / term matching
2. **Dense retrieval (vector)** — great for semantic / paraphrase matching
3. **Ensemble** — merges both result lists with configurable weights
4. **Reranking** — uses a cross-encoder to re-score and reorder the combined results

This is what production RAG systems typically look like.

In [ ]:
!uv add -q rank-bm25 flashrank langchain-classic

/Users/danhmac/Documents/agentic_bootcamp/.venv/bin/python: No module named uv
Note: you may need to restart the kernel to use updated packages.


In [18]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.document_compressors.flashrank_rerank import FlashrankRerank

# --- BM25 (sparse) retriever over the same documents ---
bm25_retriever = BM25Retriever.from_documents(documents, k=10)
logger.info("BM25 sparse retriever created over %d documents", len(documents))

# --- Reuse dense (vector) retriever from Step 3, fetch more candidates for reranking ---
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

# --- Ensemble: combine BM25 + dense with 40/60 weighting ---
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.4, 0.6],
)
logger.info("Ensemble retriever created (BM25=0.4, dense=0.6)")

# --- Reranker: FlashRank cross-encoder re-scores the ensemble output ---
reranker = FlashrankRerank(top_n=5)
hybrid_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=ensemble_retriever,
)
logger.info("Hybrid retriever with FlashRank reranking ready (top_n=5)")

# Quick comparison: same query through vanilla vs hybrid
test_query = "Who was the commander of the mission?"
print("=== Dense-only retriever (top 3) ===")
for i, doc in enumerate(retriever.invoke(test_query)[:3]):
    print(f"  [{i+1}] {doc.page_content}...\n")

print("=== Hybrid retriever with reranking (top 3) ===")
for i, doc in enumerate(hybrid_retriever.invoke(test_query)[:3]):
    print(f"  [{i+1}] {doc.page_content}...\n")

2026-04-05 15:48:43,587 [INFO] rag_lab - BM25 sparse retriever created over 33 documents
2026-04-05 15:48:43,587 [INFO] rag_lab - Ensemble retriever created (BM25=0.4, dense=0.6)
2026-04-05 15:48:43,791 [INFO] rag_lab - Hybrid retriever with FlashRank reranking ready (top_n=5)


=== Dense-only retriever (top 3) ===


2026-04-05 15:48:44,338 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


  [1] ### The Crew: Six Pioneers of a New Frontier

The *Aurora-1* crew consists of six individuals, each representing a different discipline and, in many ways, a different facet of humanity itself.

* **Commander Elena Navarro (Spain)** — A veteran astronaut and aerospace engineer, Navarro has led multiple lunar missions. Known for her calm precision, she was the natural choice to command humanity’s first Martian landing....

  [2] ### The Mission: A Journey Decades in the Making

The mission, officially named *Aurora-1*, is the culmination of over thirty years of international collaboration, technological innovation, and relentless ambition. Unlike previous robotic missions, which paved the way with data and imagery, *Aurora-1* represents humanity’s first physical presence on another planet....

  [3] * **Dr. Malik Okoye (Nigeria)** — Astrobiologist and mission science lead. His research focuses on extremophiles—organisms that survive in harsh environments—making him uniquely suited 

2026-04-05 15:48:44,668 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


  [1] ### The Crew: Six Pioneers of a New Frontier

The *Aurora-1* crew consists of six individuals, each representing a different discipline and, in many ways, a different facet of humanity itself.

* **Commander Elena Navarro (Spain)** — A veteran astronaut and aerospace engineer, Navarro has led multiple lunar missions. Known for her calm precision, she was the natural choice to command humanity’s first Martian landing....

  [2] The chosen landing site, Jezero Crater, was selected for its scientific significance. Once believed to be an ancient lakebed, it offers the highest probability of uncovering evidence of past microbial life. For scientists back on Earth, this mission is not only about exploration—it is about answering one of humanity’s oldest questions: Are we alone?...

  [3] ### The Mission: A Journey Decades in the Making

The mission, officially named *Aurora-1*, is the culmination of over thirty years of international collaboration, technological innovation, and relentl

In [19]:
hybrid_retriever_tool = create_retriever_tool(
    hybrid_retriever,
    name="mars_report_hybrid_search",
    description=(
        "Search the Mars landing report using hybrid retrieval (keyword + semantic + reranking). "
        "Use this tool to find facts before answering any question about the Aurora-1 mission."
    ),
)

hybrid_agent = create_agent(
    model=llm,
    tools=[hybrid_retriever_tool],
    system_prompt=SYSTEM_PROMPT,
)

logger.info("Hybrid RAG agent created")

# Run the same test questions from Step 4 so we can visually compare
for question in test_questions:
    print(f"\n{'=' * 70}")
    print(f"Q: {question}")
    print(f"{'=' * 70}")
    response = hybrid_agent.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content
    print(f"A: {answer}")

2026-04-05 15:50:28,682 [INFO] rag_lab - Hybrid RAG agent created



Q: Who was the first person to step on Mars?


2026-04-05 15:50:30,255 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-05 15:50:31,360 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-05 15:50:33,176 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: The retrieved information does not specify the name of the first person to step on Mars during the *Aurora-1* mission. It mentions the experience of walking on Mars and includes a quote from Dr. Tanaka, but does not identify who made the first step.

Q: What was the name of the mission and when did it launch?


2026-04-05 15:50:34,450 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-05 15:50:34,654 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-05 15:50:36,053 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

A: The *Aurora-1* mission crew consisted of six members, each representing different nationalities and disciplines. One of the crew members identified is:

- **Commander Elena Navarro (Spain)**

Unfortunately, the retrieved documents do not provide the names or nationalities of the other five crew members.

Q: What scientific discoveries were made on Mars?


2026-04-05 15:50:42,131 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-05 15:50:42,418 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-05 15:50:45,066 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: The *Aurora-1* mission focused on the Jezero Crater, which is believed to have once been an ancient lakebed. This site was chosen for its scientific significance, as it offers a high probability of uncovering evidence of past microbial life on Mars. The mission aims to answer one of humanity's oldest questions: Are we alone? 

While specific scientific discoveries from the mission are not detailed in the retrieved information, the emphasis on the potential for finding evidence of past life indicates that significant discoveries related to Mars' history and the possibility of life are anticipated.

Q: What challenges did the crew face living on Mars?


2026-04-05 15:50:45,968 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-05 15:50:46,338 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-05 15:50:49,479 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: The crew faced several challenges while living on Mars, including:

1. **Extreme Temperatures**: Mars experiences significant temperature fluctuations, with relatively mild conditions during the day and extreme cold at night.

2. **Dust Storms**: Although the crew had not yet encountered dust storms, these remain a persistent threat on Mars.

3. **Physical Constraints**: The crew had to navigate the Martian terrain, which consists of fine dust and jagged rocks, while wearing bulky suits designed to protect against extreme temperatures, radiation, and the near-vacuum atmosphere. This made every movement deliberate and constrained.

4. **Isolation and Silence**: The environment on Mars is characterized by overwhelming silence, with no wind or familiar sounds, contributing to a sense of isolation.

These challenges highlight the difficulties of adapting to life on another planet while conducting scientific research.


## Step 7: Comprehensive Evaluation — Vanilla vs Hybrid

Re-run the evaluation with **additional retrieval metrics** and compare both pipelines side-by-side.

| Metric | What it measures |
|---|---|
| **Hit Rate** | Did *any* relevant chunk appear in top-k? |
| **MRR** | How high was the *first* relevant chunk ranked? |
| **Precision@k** | What fraction of the top-k results are relevant? |
| **Recall@k** | What fraction of *all* relevant chunks appear in top-k? |
| **nDCG@k** | Are relevant chunks ranked higher than irrelevant ones? (position-aware) |
| **Faithfulness** | Does the answer only use info from retrieved context? (LLM judge, 1-5) |
| **Correctness** | Does the answer match the ground truth? (LLM judge, 1-5) |

In [20]:
import math


def compute_ndcg(relevance_scores: list[int], k: int) -> float:
    """Compute nDCG@k from a list of binary relevance labels."""
    dcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(relevance_scores[:k]))
    ideal = sorted(relevance_scores, reverse=True)[:k]
    idcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def evaluate_rag_pipeline(
    eval_data: list[dict],
    ret,
    ag,
    all_docs: list[Document],
    judge: ChatOpenAI,
    k: int = 5,
    label: str = "",
) -> list[dict]:
    """Run full retrieval + generation evaluation and return per-question metrics."""
    results = []

    for idx, item in enumerate(eval_data):
        question = item["question"]
        ground_truth = item["ground_truth"]
        keywords = item["keywords"]

        # --- Retrieval ---
        retrieved_docs = ret.invoke(question)[:k]
        retrieved_texts = [doc.page_content for doc in retrieved_docs]
        combined_context = "\n---\n".join(retrieved_texts)

        relevance = [
            1 if any(kw.lower() in txt.lower() for kw in keywords) else 0
            for txt in retrieved_texts
        ]

        total_relevant_in_corpus = sum(
            1 for doc in all_docs
            if any(kw.lower() in doc.page_content.lower() for kw in keywords)
        )

        hit = 1 if any(r == 1 for r in relevance) else 0

        rr = 0.0
        for rank, rel in enumerate(relevance, start=1):
            if rel == 1:
                rr = 1.0 / rank
                break

        precision_k = sum(relevance) / k if k > 0 else 0.0
        recall_k = sum(relevance) / total_relevant_in_corpus if total_relevant_in_corpus > 0 else 0.0
        ndcg_k = compute_ndcg(relevance, k)

        # --- Generation ---
        response = ag.invoke({"messages": [{"role": "user", "content": question}]})
        answer = response["messages"][-1].content

        # --- LLM-as-judge ---
        judge_prompt = JUDGE_TEMPLATE.format(
            question=question,
            ground_truth=ground_truth,
            context=combined_context[:3000],
            answer=answer,
        )
        judge_resp = judge.invoke(judge_prompt)
        judge_text = judge_resp.content

        faith, correct = 0, 0
        for line in judge_text.strip().split("\n"):
            if line.lower().startswith("faithfulness"):
                try:
                    faith = int(line.split(":")[1].strip())
                except (ValueError, IndexError):
                    pass
            elif line.lower().startswith("correctness"):
                try:
                    correct = int(line.split(":")[1].strip())
                except (ValueError, IndexError):
                    pass

        results.append({
            "question": question,
            "answer": answer,
            "hit": hit,
            "rr": rr,
            "precision_k": precision_k,
            "recall_k": recall_k,
            "ndcg_k": ndcg_k,
            "faithfulness": faith,
            "correctness": correct,
        })

        logger.info(
            "[%s %d/%d] %s | hit=%d p@k=%.2f r@k=%.2f ndcg=%.2f faith=%d corr=%d",
            label, idx + 1, len(eval_data), question[:30],
            hit, precision_k, recall_k, ndcg_k, faith, correct,
        )

    return results


logger.info("Evaluation function ready")

2026-04-05 16:24:19,039 [INFO] rag_lab - Evaluation function ready


In [21]:
logger.info("Running evaluation on VANILLA (dense-only) pipeline...")
vanilla_eval = evaluate_rag_pipeline(
    eval_dataset, retriever, agent, documents, judge_llm, k=5, label="vanilla",
)

logger.info("Running evaluation on HYBRID (BM25 + dense + rerank) pipeline...")
hybrid_eval = evaluate_rag_pipeline(
    eval_dataset, hybrid_retriever, hybrid_agent, documents, judge_llm, k=5, label="hybrid",
)

logger.info("Both evaluations complete")

2026-04-05 16:24:24,080 [INFO] rag_lab - Running evaluation on VANILLA (dense-only) pipeline...
2026-04-05 16:24:25,398 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-05 16:24:27,330 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-05 16:24:27,667 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-05 16:24:29,564 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-05 16:24:30,258 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-05 16:24:30,261 [INFO] rag_lab - [vanilla 1/10] Who was the commander of the A | hit=1 p@k=0.20 r@k=0.50 ndcg=0.63 faith=5 corr=5
2026-04-05 16:24:31,514 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-05 16:24:33,496 [INFO] httpx - HTTP Request: POST https://api.

In [22]:
def summarize(results: list[dict]) -> dict:
    n = len(results)
    return {
        "Hit Rate": sum(r["hit"] for r in results) / n,
        "MRR": sum(r["rr"] for r in results) / n,
        "Precision@k": sum(r["precision_k"] for r in results) / n,
        "Recall@k": sum(r["recall_k"] for r in results) / n,
        "nDCG@k": sum(r["ndcg_k"] for r in results) / n,
        "Faithfulness": sum(r["faithfulness"] for r in results) / n,
        "Correctness": sum(r["correctness"] for r in results) / n,
    }


vanilla_summary = summarize(vanilla_eval)
hybrid_summary = summarize(hybrid_eval)

# --- Per-question comparison ---
print(f"\n{'=' * 100}")
print(f"{'PER-QUESTION COMPARISON':^100}")
print(f"{'=' * 100}")
header = f"{'Question':<40} | {'Hit':^7} | {'P@k':^11} | {'R@k':^11} | {'nDCG':^11} | {'Corr':^9}"
print(f"\n{header}")
print(f"{'-' * 40}-+-{'-' * 7}-+-{'-' * 11}-+-{'-' * 11}-+-{'-' * 11}-+-{'-' * 9}")

for v, h in zip(vanilla_eval, hybrid_eval):
    q = (v["question"][:37] + "...") if len(v["question"]) > 40 else v["question"]
    print(
        f"{q:<40} | "
        f"{'V' if v['hit'] else '-':>1} / {'H' if h['hit'] else '-':>1}   | "
        f"{v['precision_k']:.2f}/{h['precision_k']:.2f}  | "
        f"{v['recall_k']:.2f}/{h['recall_k']:.2f}  | "
        f"{v['ndcg_k']:.2f}/{h['ndcg_k']:.2f}  | "
        f"{v['correctness']}/{h['correctness']}      "
    )

# --- Aggregate comparison ---
print(f"\n\n{'=' * 60}")
print(f"{'AGGREGATE COMPARISON':^60}")
print(f"{'=' * 60}")
print(f"\n{'Metric':<20} {'Vanilla':>12} {'Hybrid':>12} {'Delta':>10}")
print(f"{'-' * 20} {'-' * 12} {'-' * 12} {'-' * 10}")

for metric in vanilla_summary:
    v_val = vanilla_summary[metric]
    h_val = hybrid_summary[metric]
    delta = h_val - v_val
    sign = "+" if delta > 0 else ""
    if metric in ("Faithfulness", "Correctness"):
        print(f"{metric:<20} {v_val:>10.2f}/5 {h_val:>10.2f}/5 {sign}{delta:>8.2f}")
    else:
        print(f"{metric:<20} {v_val:>11.1%} {h_val:>11.1%} {sign}{delta:>8.1%}")

print(f"{'=' * 60}")
print(f"\n  Chunking strategy used: {CHUNKING_STRATEGY}")
print(f"  Vanilla = dense vector retrieval only")
print(f"  Hybrid  = BM25 + dense ensemble + FlashRank reranking")


                                      PER-QUESTION COMPARISON                                       

Question                                 |   Hit   |     P@k     |     R@k     |    nDCG     |   Corr   
-----------------------------------------+---------+-------------+-------------+-------------+----------
Who was the commander of the Aurora-1... | V / H   | 0.20/0.20  | 0.50/0.50  | 0.63/0.63  | 5/5      
When did humans first land on Mars?      | V / -   | 0.20/0.00  | 1.00/0.00  | 1.00/0.00  | 5/5      
What was the name of the lander?         | V / H   | 0.40/0.40  | 1.00/1.00  | 0.92/0.92  | 1/5      
What is the gravity on Mars compared ... | V / H   | 0.20/0.20  | 1.00/1.00  | 1.00/1.00  | 5/5      
What was Dr. Malik Okoye's role on th... | V / H   | 0.20/0.20  | 1.00/1.00  | 1.00/0.50  | 5/5      
What were the early scientific findin... | V / -   | 0.20/0.00  | 1.00/0.00  | 0.39/0.00  | 5/5      
Where did the Aurora-1 mission land o... | - / -   | 0.00/0.00  | 0.00/0.00

## TODO

- [ ] **Log evaluation metrics to LangSmith / Langfuse** — Integrate with an observability platform so you can track retrieval and generation metrics across runs, compare experiments, and share results with your team.
- [ ] **Try with other methods to see the difference** — Swap out components (e.g. different chunking strategies, embedding models, distance metrics, retriever weights, reranker models) and re-run the evaluation to understand how each choice affects quality.
- [ ] **Add more test cases to see how the RAG performs** — Expand `eval_dataset` with harder questions (multi-hop, negation, out-of-scope) to stress-test retrieval and generation quality.
- [ ] **Add more document to RAG** - Add spaceship doc to the knowledge and try to ask some more questions related to that doc to see how the agent will answer.